# Phase 5 â€” Embedding v5: Training Strategy & Architecture Improvements

**Goal:** Improve embedding pol_match beyond current ceiling (~82%) via training strategy and architecture changes.

**Baseline:** MAMS (polonly + MAMS combined, tau=0.07, 1-layer projection) = test pol_match@5 = **0.822**

**3 Experiments (separated to measure individual effects):**
1. **Exp tau012:** tau=0.07 â†’ 0.12 (same architecture) â€” tests if softer softmax reduces cosine collapse
2. **Exp deep_proj:** 1-layer â†’ 2-layer projection (same tau=0.07) â€” tests if more capacity helps polarity separation
3. **Exp combined:** tau=0.12 + 2-layer projection â€” tests combined effect

**Input:** `lcminhc/semeval-2014-absa-restaurant`

**Output:** Best embedding checkpoint + FAISS index â†’ upload as `p5-embed-v5`

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil
import glob

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire dataset
import glob
import os
import shutil

KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-2014-absa-restaurant',
                  '/kaggle/input/semeval-2014-task-4-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-2014-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
if not KAGGLE_INPUT:
    matches = glob.glob('/kaggle/input/*restaurant*')
    if matches:
        KAGGLE_INPUT = matches[0]

assert KAGGLE_INPUT, f"Dataset not found. /kaggle/input contains: {os.listdir('/kaggle/input')}"
print(f"Input: {KAGGLE_INPUT}")
print("Files:", os.listdir(KAGGLE_INPUT))

# Copy processed data
os.makedirs('data/processed', exist_ok=True)
for f2 in ['contrastive_triplets.jsonl', 'classification.jsonl',
          'sentiment_records.jsonl']:
    src = f'{KAGGLE_INPUT}/{f2}'
    dst = f'data/processed/{f2}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        with open(dst) as fp:
            n = sum(1 for _ in fp)
        print(f'{f2}: {n} records')
    else:
        print(f'WARNING: {f2} not found in dataset')

# Wire SemEval 2014 XMLs
os.makedirs('SemEval-2014', exist_ok=True)
for xml in ['Restaurants_Train.xml', 'Restaurants_Test_Gold.xml']:
    src = f'{KAGGLE_INPUT}/{xml}'
    if os.path.exists(src):
        shutil.copy(src, f'SemEval-2014/{xml}')
if os.path.exists('SemEval-2014/Restaurants_Train.xml'):
    print('SemEval 2014 XMLs wired.')

# Clone MAMS dataset
if not os.path.exists('data/mams'):
    !git clone --depth 1 https://github.com/siat-nlp/MAMS-for-ABSA.git data/mams
    print(f'MAMS cloned: {os.path.exists("data/mams/data/MAMS-ACSA/raw/train.xml")}')
else:
    print('MAMS already present')


In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 0.5 Prepare Shared Data

All 3 experiments use the same MAMS polonly triplets (same as MAMS baseline).

In [ ]:
# Generate polonly+MAMS triplets (shared across all experiments)
!python scripts/01_prepare_data.py --include_mams

# Verify
with open('data/processed/contrastive_triplets_polonly_mams.jsonl') as f:
    n = sum(1 for _ in f)
print(f'\nShared triplets: {n} records')

---
# Shared Evaluation Function

Reusable eval function â€” builds FAISS from SemEval train only, measures test pol_match@5.

In [ ]:
import numpy as np, faiss as _faiss
from collections import defaultdict as _dd
from transformers import AutoTokenizer as _AT
from src.embedding.model import ContrastiveEmbedder as _CE
from src.retrieval.encoder import encode_records as _enc

def eval_embedding(ckpt_path, label='', proj_depth=1):
    """Evaluate embedding: build FAISS from SemEval train, measure pol_match@5 on test."""
    all_cls = [json.loads(l) for l in open('data/processed/classification.jsonl')]
    train_recs = [r for r in all_cls if r['split'] == 'train']
    test_recs = [r for r in all_cls if r['split'] == 'test']

    tok = _AT.from_pretrained('microsoft/deberta-v3-base')
    emb = _CE(model_name='microsoft/deberta-v3-base', proj_dim=256, proj_depth=proj_depth)
    emb.load_state_dict(torch.load(ckpt_path, map_location='cuda'), strict=False)
    emb.to('cuda').eval()

    # Encode and build FAISS from SemEval train only
    train_vecs = _enc(train_recs, emb, tok, max_length=128, batch_size=64, device='cuda')
    _faiss.normalize_L2(train_vecs)
    index = _faiss.IndexFlatIP(train_vecs.shape[1])
    index.add(train_vecs)
    train_meta = [{'polarity': r['polarity'], 'aspect_category': r.get('aspect_category', r.get('category'))} for r in train_recs]

    # Encode test
    test_vecs = _enc(test_recs, emb, tok, max_length=128, batch_size=64, device='cuda')
    _faiss.normalize_L2(test_vecs)
    K = 5
    D, I = index.search(test_vecs, K)

    # pol_match@5
    test_pm = _dd(list)
    for i, rec in enumerate(test_recs):
        q_pol = rec['polarity']
        matches = sum(1 for j in I[i] if j >= 0 and train_meta[j]['polarity'] == q_pol)
        test_pm[q_pol].append(matches / K)

    # Train self-eval
    D_tr, I_tr = index.search(train_vecs, K + 1)
    train_pm = _dd(list)
    for i, rec in enumerate(train_recs):
        neighbors = [j for j in I_tr[i] if j != i][:K]
        matches = sum(1 for j in neighbors if j >= 0 and train_meta[j]['polarity'] == rec['polarity'])
        train_pm[rec['polarity']].append(matches / K)

    # Print
    scores = D[D > -1e9]
    overall = np.mean([m for ms in test_pm.values() for m in ms])
    train_overall = np.mean([m for ms in train_pm.values() for m in ms])

    print(f'\n{"="*60}')
    print(f'{label} \u2014 TEST pol_match@{K}')
    print(f'{"="*60}')
    print(f'{"Polarity":<12} {"n":<6} {"Test":<8} {"Train":<8}')
    print(f'{"-"*36}')
    for pol in ['positive', 'negative', 'neutral']:
        t = np.mean(test_pm[pol]) if test_pm[pol] else 0
        tr = np.mean(train_pm[pol]) if train_pm[pol] else 0
        print(f'{pol:<12} {len(test_pm[pol]):<6} {t:<8.3f} {tr:<8.3f}')
    print(f'{"-"*36}')
    print(f'{"Overall":<12} {len(test_recs):<6} {overall:<8.3f} {train_overall:<8.3f}')
    print(f'Cosine: mean={scores.mean():.4f}, std={scores.std():.4f}')

    del emb, tok
    gc.collect(); torch.cuda.empty_cache()
    return {'overall': overall, 'positive': np.mean(test_pm['positive']),
            'negative': np.mean(test_pm['negative']), 'neutral': np.mean(test_pm['neutral']),
            'cosine_mean': float(scores.mean()), 'cosine_std': float(scores.std())}

print('eval_embedding() defined.')

---
# Experiment 1: tau=0.12 (Architecture unchanged)

**Hypothesis:** tau=0.07 is too small â†’ softmax too sharp â†’ cosine collapse (~0.993). Increasing tau to 0.12 should spread embeddings wider, making FAISS scores more discriminative.

**Changes vs baseline:** ONLY `tau: 0.07 â†’ 0.12`. Same 1-layer projection, same data.

In [ ]:
gc.collect(); torch.cuda.empty_cache()
print('='*60)
print('EXP 1: tau=0.12 (same architecture as baseline)')
print('='*60)

# Stage 1: Train with random triplets
!python scripts/02_train_embedding.py --config configs/embedding_2014_tau012_s1.yaml

# Hard negative mining
!python scripts/build_hard_triplets.py \
    --embedding_ckpt checkpoints/embedding_2014_tau012_s1/best.pt \
    --cls_path data/processed/classification.jsonl \
    --out_path data/processed/hard_contrastive_triplets_tau012.jsonl \
    --no_neg2

# Stage 2: Train with hard triplets
!python scripts/02_train_embedding.py --config configs/embedding_2014_tau012_s2.yaml \
    --resume_from checkpoints/embedding_2014_tau012_s1/best.pt

# Evaluate
results_tau012 = eval_embedding(
    'checkpoints/embedding_2014_tau012_s2/best.pt',
    'Exp1: tau=0.12 (1-layer proj)',
    proj_depth=1
)

---
# Experiment 2: Deeper Projection Head (tau=0.07 unchanged)

**Hypothesis:** 1-layer projection (768â†’256) has limited capacity to remap CLS space to polarity-aware space. 2-layer (768â†’512â†’256) with GELU+LayerNorm adds non-linearity for better polarity separation.

**Changes vs baseline:** ONLY `proj_depth: 1 â†’ 2`. Same tau=0.07, same data.

In [ ]:
gc.collect(); torch.cuda.empty_cache()
print('='*60)
print('EXP 2: Deeper projection (tau=0.07, 2-layer proj)')
print('='*60)

# Stage 1: Train with random triplets
!python scripts/02_train_embedding.py --config configs/embedding_2014_deep_s1.yaml

# Hard negative mining (need --proj_depth for correct model loading)
!python scripts/build_hard_triplets.py \
    --embedding_ckpt checkpoints/embedding_2014_deep_s1/best.pt \
    --cls_path data/processed/classification.jsonl \
    --out_path data/processed/hard_contrastive_triplets_deep.jsonl \
    --no_neg2 \
    --proj_depth 2

# Stage 2: Train with hard triplets
!python scripts/02_train_embedding.py --config configs/embedding_2014_deep_s2.yaml \
    --resume_from checkpoints/embedding_2014_deep_s1/best.pt

# Evaluate
results_deep = eval_embedding(
    'checkpoints/embedding_2014_deep_s2/best.pt',
    'Exp2: deep proj (2-layer, tau=0.07)',
    proj_depth=2
)

---
# Experiment 3: tau=0.12 + Deeper Projection (Combined)

**Hypothesis:** Combining both improvements â€” softer softmax prevents collapse while deeper projection increases capacity.

**Changes vs baseline:** `tau: 0.12` + `proj_depth: 2`.

In [ ]:
gc.collect(); torch.cuda.empty_cache()
print('='*60)
print('EXP 3: tau=0.12 + deep projection (combined)')
print('='*60)

# Stage 1: Train with random triplets
!python scripts/02_train_embedding.py --config configs/embedding_2014_tau012_deep_s1.yaml

# Hard negative mining
!python scripts/build_hard_triplets.py \
    --embedding_ckpt checkpoints/embedding_2014_tau012_deep_s1/best.pt \
    --cls_path data/processed/classification.jsonl \
    --out_path data/processed/hard_contrastive_triplets_tau012_deep.jsonl \
    --no_neg2 \
    --proj_depth 2

# Stage 2: Train with hard triplets
!python scripts/02_train_embedding.py --config configs/embedding_2014_tau012_deep_s2.yaml \
    --resume_from checkpoints/embedding_2014_tau012_deep_s1/best.pt

# Evaluate
results_combined = eval_embedding(
    'checkpoints/embedding_2014_tau012_deep_s2/best.pt',
    'Exp3: tau=0.12 + deep proj',
    proj_depth=2
)

---
# Final Comparison & Save Best

In [ ]:
# === COMPARISON TABLE ===
baseline = {'overall': 0.822, 'positive': 0.905, 'negative': 0.695, 'neutral': 0.545,
            'cosine_mean': 0.9929, 'cosine_std': 0.0}

all_results = {'MAMS baseline (tau=0.07, 1L)': baseline}
if 'results_tau012' in dir(): all_results['Exp1: tau=0.12 (1L)'] = results_tau012
if 'results_deep' in dir(): all_results['Exp2: deep proj (2L, tau=0.07)'] = results_deep
if 'results_combined' in dir(): all_results['Exp3: tau=0.12 + deep (2L)'] = results_combined

print(f'\n{"="*80}')
print(f'EMBEDDING v5 EXPERIMENT COMPARISON â€” TEST pol_match@5')
print(f'{"="*80}')
print(f'{"Experiment":<30} {"Overall":<9} {"Pos":<8} {"Neg":<8} {"Neu":<8} {"CosMean":<9} {"CosStd":<8}')
print(f'{"-"*80}')
for name, r in all_results.items():
    cos_m = r.get('cosine_mean', 0)
    cos_s = r.get('cosine_std', 0)
    print(f'{name:<30} {r["overall"]:<9.3f} {r["positive"]:<8.3f} {r["negative"]:<8.3f} {r["neutral"]:<8.3f} {cos_m:<9.4f} {cos_s:<8.4f}')

# Delta vs baseline
print(f'\n{"="*80}')
print(f'DELTA vs BASELINE')
print(f'{"="*80}')
print(f'{"Experiment":<30} {"dOverall":<10} {"dPos":<8} {"dNeg":<8} {"dNeu":<8}')
print(f'{"-"*80}')
for name, r in all_results.items():
    if name == 'MAMS baseline (tau=0.07, 1L)': continue
    do = r['overall'] - baseline['overall']
    dp = r['positive'] - baseline['positive']
    dn = r['negative'] - baseline['negative']
    dne = r['neutral'] - baseline['neutral']
    print(f'{name:<30} {do:+<10.3f} {dp:+<8.3f} {dn:+<8.3f} {dne:+<8.3f}')

# Find best
results_only = {k: v for k, v in all_results.items() if k != 'MAMS baseline (tau=0.07, 1L)'}
if results_only:
    best_name = max(results_only.keys(), key=lambda k: results_only[k]['overall'])
    best = results_only[best_name]
    print(f'\n>>> BEST: {best_name} (overall={best["overall"]:.3f}, delta={best["overall"]-baseline["overall"]:+.3f})')

In [ ]:
# === SAVE BEST EMBEDDING ===
output_dir = '/kaggle/working/outputs_p5_embed_v5'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

# Map experiment name to checkpoint and proj_depth
ckpt_map = {
    'Exp1: tau=0.12 (1L)': ('checkpoints/embedding_2014_tau012_s2/best.pt', 1),
    'Exp2: deep proj (2L, tau=0.07)': ('checkpoints/embedding_2014_deep_s2/best.pt', 2),
    'Exp3: tau=0.12 + deep (2L)': ('checkpoints/embedding_2014_tau012_deep_s2/best.pt', 2),
}

if results_only:
    best_ckpt, best_proj_depth = ckpt_map[best_name]
    if os.path.exists(best_ckpt):
        shutil.copy(best_ckpt, f'{output_dir}/embedding_best.pt')
        shutil.copy(best_ckpt, f'{output_dir}/embedding_v5_best.pt')
        
        # Save metadata
        with open(f'{output_dir}/best_config.json', 'w') as f:
            json.dump({
                'experiment': best_name,
                'proj_depth': best_proj_depth,
                'results': best,
                'baseline': baseline,
            }, f, indent=2)
        print(f'Saved best embedding: {best_ckpt}')
        print(f'proj_depth={best_proj_depth}')

        # Build FAISS index from best embedding
        gc.collect(); torch.cuda.empty_cache()
        !python scripts/03_build_index.py \
            --embedding_ckpt {best_ckpt} \
            --input data/processed/sentiment_records.jsonl \
            --out_dir indexes/ \
            --proj_depth {best_proj_depth}
        for f_name in ['train.faiss', 'train_metadata.jsonl', 'train_vectors.npy']:
            src = f'indexes/{f_name}'
            if os.path.exists(src):
                shutil.copy(src, f'{output_dir}/{f_name}')
        print('FAISS index saved.')

# Save all experiment logs
for log in glob.glob('logs/embedding_2014_tau012*.jsonl') + \
          glob.glob('logs/embedding_2014_deep*.jsonl'):
    shutil.copy(log, f'{output_dir}/logs/')

# Also save all 3 checkpoints for comparison
for exp_name, (ckpt, pd) in ckpt_map.items():
    if os.path.exists(ckpt):
        short = exp_name.split(':')[0].strip().lower().replace(' ', '_')
        shutil.copy(ckpt, f'{output_dir}/embedding_{short}.pt')

# Backup
shutil.make_archive('/kaggle/working/outputs_p5_embed_v5_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_embed_v5')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_embed_v5_backup.zip') / 1e6
print(f'\nBackup zip: {size_mb:.1f} MB')
print('\nDONE! Upload outputs_p5_embed_v5 as Kaggle dataset p5-embed-v5')